# Error Analysis

Systematic error analysis across all models (decoders: zero-shot & chain-of-thought; encoders: fine-tuned) on two datasets (**CSC** and **MultiPICo**) for two input types: `last_u1_message` and `conversation`.

Analysis covers:
1. **Data loading** — collect all predictions into a unified DataFrame
2. **All-models-wrong** — examples where every model fails (both input types)
3. **Input-type disagreement** — examples where models fail on `last_u1_message` but not `conversation` (and vice-versa)
4. **Error rate by label** — bias towards false positives / false negatives
5. **Model-specific error patterns** — which model makes unique errors
6. **Disagreement matrix** — pairwise model disagreement
7. **Qualitative inspection** — display dialogue text for hard cases


In [1]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations

BASE = Path(r"c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\output")

INPUT_TYPES = ["last_u1_message", "conversation"]
COMMON_SAMPLE_INPUT_TYPES = ["context_reply", *INPUT_TYPES]
DATASETS   = ["CSC", "MultiPICo"]

# ── DECODER files ─────────────────────────────────────────────────────────────
DECODERS = {
    "Gemma4-E4B":    BASE / "decoders" / "google_gemma-4-E4B-it",
    "Ministral3-8B":  BASE / "decoders" / "mistralai_Ministral-3-8B-Instruct-2512",
    "Qwen3.5-9B":   BASE / "decoders" / "Qwen_Qwen3.5-9B",
}
DECODER_PROMPTS = ["zero_shot", "chain_of_thought"]

# ── ENCODER files ─────────────────────────────────────────────────────────────
ENCODERS = {
    "XLM-RoBERTa-large": BASE / "encoders" / "FacebookAI_xlm-roberta-large",
    "EuroBERT-210m":     BASE / "encoders" / "EuroBERT_EuroBERT-210m",
}

# ──────────────────────────────────────────────────────────────────────────────
records = []

for model_name, folder in DECODERS.items():
    for prompt in DECODER_PROMPTS:
        for dataset in DATASETS:
            for inp in COMMON_SAMPLE_INPUT_TYPES:
                # find matching file
                candidates = list(folder.glob(f"{prompt}_*_{dataset}_{inp}.csv"))
                if not candidates:
                    continue
                df = pd.read_csv(candidates[0], usecols=["unique_id", "model_answer", "label"])
                df = df[df["model_answer"].notna()]
                df["model"]      = f"{model_name}_{prompt}"
                df["dataset"]    = dataset
                df["input_type"] = inp
                records.append(df)

for model_name, enc_base in ENCODERS.items():
    for dataset in DATASETS:
        for inp in COMMON_SAMPLE_INPUT_TYPES:
            pred_dir = enc_base / dataset / "human_heldout_f1" / inp / "predictions"
            candidates = [
                path for path in pred_dir.glob("*.csv")
                if not path.stem.endswith("_human_majority")
            ]
            if not candidates:
                continue
            csv_path = str(candidates[0].resolve())
            long_path_prefix = chr(92) * 2 + "?" + chr(92)
            if os.name == "nt" and not csv_path.startswith(long_path_prefix):
                csv_path = long_path_prefix + csv_path
            df = pd.read_csv(csv_path, usecols=["unique_id", "model_answer", "label"])
            df = df[df["model_answer"].notna()]
            df["model"]      = f"{model_name}_fine_tuning"
            df["dataset"]    = dataset
            df["input_type"] = inp
            records.append(df)

all_preds = pd.concat(records, ignore_index=True)
all_preds["unique_id"] = all_preds["unique_id"].astype("string")
all_preds["model_answer"] = pd.to_numeric(all_preds["model_answer"], errors="coerce")
all_preds["label"]        = pd.to_numeric(all_preds["label"],        errors="coerce")
all_preds = all_preds.dropna(subset=["model_answer", "label"])
all_preds["model_answer"] = all_preds["model_answer"].astype(int)
all_preds["label"]        = all_preds["label"].astype(int)
all_preds["correct"]      = (all_preds["model_answer"] == all_preds["label"]).astype(int)
all_preds["error_type"]   = all_preds.apply(
    lambda r: "FP" if r["model_answer"] == 1 and r["label"] == 0
         else ("FN" if r["model_answer"] == 0 and r["label"] == 1 else "OK"),
    axis=1
)

# Use the same per-dataset common sample as Table 4: every system must
# have a valid prediction for every reported input configuration.
models = all_preds["model"].unique().tolist()
common_ids_by_dataset = {}
for dataset in DATASETS:
    valid_id_sets = []
    for model in models:
        for inp in COMMON_SAMPLE_INPUT_TYPES:
            ids = set(
                all_preds.loc[
                    (all_preds["dataset"] == dataset)
                    & (all_preds["model"] == model)
                    & (all_preds["input_type"] == inp),
                    "unique_id",
                ].tolist()
            )
            if not ids:
                raise ValueError(
                    f"No valid predictions for {dataset} | {model} | {inp}"
                )
            valid_id_sets.append(ids)
    common_ids_by_dataset[dataset] = set.intersection(*valid_id_sets)
    print(
        f"Common sample [{dataset}]: "
        f"{len(common_ids_by_dataset[dataset])} instances"
    )

all_preds = all_preds[all_preds["input_type"].isin(INPUT_TYPES)].copy()
all_preds = all_preds[
    all_preds.apply(
        lambda row: row["unique_id"] in common_ids_by_dataset[row["dataset"]],
        axis=1,
    )
].copy()

models = all_preds["model"].unique().tolist()
print(f"Models loaded: {models}")
print(f"Total prediction rows: {len(all_preds)}")
all_preds.head(3)


Common sample [CSC]: 418 instances
Common sample [MultiPICo]: 192 instances
Models loaded: ['Gemma4-E4B_zero_shot', 'Gemma4-E4B_chain_of_thought', 'Ministral3-8B_zero_shot', 'Ministral3-8B_chain_of_thought', 'Qwen3.5-9B_zero_shot', 'Qwen3.5-9B_chain_of_thought', 'XLM-RoBERTa-large_fine_tuning', 'EuroBERT-210m_fine_tuning']
Total prediction rows: 9760


,unique_id,label,model_answer,model,dataset,input_type,correct,error_type
422,206_2010,1,0,Gemma4-E4B_zero_shot,CSC,last_u1_message,0,FN
423,109_1045,0,0,Gemma4-E4B_zero_shot,CSC,last_u1_message,1,OK
424,265_2071,0,0,Gemma4-E4B_zero_shot,CSC,last_u1_message,1,OK


## 1. Overall accuracy by model, dataset, input type

In [2]:
acc_table = (
    all_preds.groupby(["dataset", "input_type", "model"])["correct"]
    .mean()
    .round(3)
    .reset_index()
    .rename(columns={"correct": "accuracy"})
    .pivot_table(index=["dataset", "model"], columns="input_type", values="accuracy")
    .reset_index()
)
print(acc_table.to_string(index=False))


  dataset                          model  conversation  last_u1_message
      CSC      EuroBERT-210m_fine_tuning         0.701            0.333
      CSC    Gemma4-E4B_chain_of_thought         0.538            0.493
      CSC           Gemma4-E4B_zero_shot         0.620            0.483
      CSC Ministral3-8B_chain_of_thought         0.502            0.505
      CSC        Ministral3-8B_zero_shot         0.409            0.383
      CSC    Qwen3.5-9B_chain_of_thought         0.565            0.574
      CSC           Qwen3.5-9B_zero_shot         0.565            0.421
      CSC  XLM-RoBERTa-large_fine_tuning         0.682            0.596
MultiPICo      EuroBERT-210m_fine_tuning         0.427            0.495
MultiPICo    Gemma4-E4B_chain_of_thought         0.604            0.443
MultiPICo           Gemma4-E4B_zero_shot         0.620            0.521
MultiPICo Ministral3-8B_chain_of_thought         0.479            0.500
MultiPICo        Ministral3-8B_zero_shot         0.365          

## 2. Error-type breakdown (FP / FN) per model × dataset × input_type

In [3]:
error_breakdown = (
    all_preds[all_preds["error_type"] != "OK"]
    .groupby(["dataset", "input_type", "model", "error_type"])
    .size()
    .reset_index(name="count")
)
pivot_err = error_breakdown.pivot_table(
    index=["dataset", "input_type", "model"],
    columns="error_type",
    values="count",
    fill_value=0
).reset_index()
print(pivot_err.to_string(index=False))


  dataset      input_type                          model   FN    FP
      CSC    conversation      EuroBERT-210m_fine_tuning 50.0  75.0
      CSC    conversation    Gemma4-E4B_chain_of_thought 43.0 150.0
      CSC    conversation           Gemma4-E4B_zero_shot 87.0  72.0
      CSC    conversation Ministral3-8B_chain_of_thought 28.0 180.0
      CSC    conversation        Ministral3-8B_zero_shot 15.0 232.0
      CSC    conversation    Qwen3.5-9B_chain_of_thought 50.0 132.0
      CSC    conversation           Qwen3.5-9B_zero_shot 53.0 129.0
      CSC    conversation  XLM-RoBERTa-large_fine_tuning 36.0  97.0
      CSC last_u1_message      EuroBERT-210m_fine_tuning  1.0 278.0
      CSC last_u1_message    Gemma4-E4B_chain_of_thought 43.0 169.0
      CSC last_u1_message           Gemma4-E4B_zero_shot 62.0 154.0
      CSC last_u1_message Ministral3-8B_chain_of_thought 35.0 172.0
      CSC last_u1_message        Ministral3-8B_zero_shot 10.0 248.0
      CSC last_u1_message    Qwen3.5-9B_chain_of

## 3. Cases where ALL models are wrong

For each (dataset, input_type) pair, find examples where every model predicts incorrectly.

In [4]:
all_wrong_by_setting = {}

for dataset in DATASETS:
    for inp in INPUT_TYPES:
        subset = all_preds[(all_preds["dataset"] == dataset) & (all_preds["input_type"] == inp)]
        n_models = subset["model"].nunique()
        # count how many models got each example wrong
        wrong_counts = (
            subset[subset["correct"] == 0]
            .groupby("unique_id")["model"]
            .nunique()
            .reset_index(name="n_models_wrong")
        )
        # keep label and total-model count
        label_map = subset[["unique_id", "label"]].drop_duplicates()
        wrong_counts = wrong_counts.merge(label_map, on="unique_id")
        all_wrong = wrong_counts[wrong_counts["n_models_wrong"] == n_models].copy()
        all_wrong["dataset"]    = dataset
        all_wrong["input_type"] = inp
        all_wrong["n_total_models"] = n_models
        key = f"{dataset}_{inp}"
        all_wrong_by_setting[key] = all_wrong
        print(f"[{key}]  total models={n_models}  examples where ALL wrong: {len(all_wrong)}")
        print(f"  label=0 (non-irony): {(all_wrong['label']==0).sum()}   label=1 (irony): {(all_wrong['label']==1).sum()}")


[CSC_last_u1_message]  total models=8  examples where ALL wrong: 27
  label=0 (non-irony): 27   label=1 (irony): 0
[CSC_conversation]  total models=8  examples where ALL wrong: 14
  label=0 (non-irony): 13   label=1 (irony): 1
[MultiPICo_last_u1_message]  total models=8  examples where ALL wrong: 9
  label=0 (non-irony): 9   label=1 (irony): 0
[MultiPICo_conversation]  total models=8  examples where ALL wrong: 6
  label=0 (non-irony): 6   label=1 (irony): 0


## 4. Input-type disagreement: wrong on `last_u1_message` but correct on `conversation` (and vice-versa)

For each model separately, find examples where the model is wrong in one input setting but right in the other.
Then aggregate to find *consistent* patterns across models.

In [5]:
# pivot so each row = (model, dataset, unique_id) with correctness per input type
pivot_inp = all_preds.pivot_table(
    index=["model", "dataset", "unique_id", "label"],
    columns="input_type",
    values="correct"
).reset_index()

# keep only rows where both input types are available for that model
pivot_inp = pivot_inp.dropna(subset=INPUT_TYPES)
pivot_inp[INPUT_TYPES] = pivot_inp[INPUT_TYPES].astype(int)

# wrong on last_u1 but right on conversation → context HELPS
pivot_inp["lu1_wrong_conv_ok"]  = ((pivot_inp["last_u1_message"] == 0) & (pivot_inp["conversation"] == 1)).astype(int)
# right on last_u1 but wrong on conversation → context HURTS
pivot_inp["lu1_ok_conv_wrong"]  = ((pivot_inp["last_u1_message"] == 1) & (pivot_inp["conversation"] == 0)).astype(int)
# wrong on BOTH
pivot_inp["both_wrong"]         = ((pivot_inp["last_u1_message"] == 0) & (pivot_inp["conversation"] == 0)).astype(int)

for dataset in DATASETS:
    sub = pivot_inp[pivot_inp["dataset"] == dataset]
    print(f"\n=== {dataset} ===")
    summary = sub.groupby("model")[["lu1_wrong_conv_ok", "lu1_ok_conv_wrong", "both_wrong"]].sum()
    print(summary.to_string())



=== CSC ===


input_type                      lu1_wrong_conv_ok  lu1_ok_conv_wrong  both_wrong
model                                                                           
EuroBERT-210m_fine_tuning                     204                 50          75
Gemma4-E4B_chain_of_thought                    83                 64         129
Gemma4-E4B_zero_shot                          122                 65          94
Ministral3-8B_chain_of_thought                 67                 68         140
Ministral3-8B_zero_shot                        45                 34         213
Qwen3.5-9B_chain_of_thought                    68                 72         110
Qwen3.5-9B_zero_shot                          120                 60         122
XLM-RoBERTa-large_fine_tuning                  78                 42          91

=== MultiPICo ===
input_type                      lu1_wrong_conv_ok  lu1_ok_conv_wrong  both_wrong
model                                                                           
EuroBERT-

### 4b. Aggregate across models: how many models show each disagreement pattern?

Find examples where the *majority* of models (or all models) show `lu1_wrong_conv_ok` vs `lu1_ok_conv_wrong`.

In [6]:
agg_disagree = (
    pivot_inp.groupby(["dataset", "unique_id", "label"])[
        ["lu1_wrong_conv_ok", "lu1_ok_conv_wrong", "both_wrong"]
    ]
    .sum()
    .reset_index()
)

n_total_models = pivot_inp["model"].nunique()

# "context helps": majority of models are wrong on last_u1 but right on conversation
context_helps = agg_disagree[agg_disagree["lu1_wrong_conv_ok"] > n_total_models / 2].copy()
# "context hurts": majority of models are right on last_u1 but wrong on conversation
context_hurts = agg_disagree[agg_disagree["lu1_ok_conv_wrong"] > n_total_models / 2].copy()

for dataset in DATASETS:
    ch = context_helps[context_helps["dataset"] == dataset]
    cu = context_hurts[context_hurts["dataset"] == dataset]
    print(f"\n=== {dataset} (total models={n_total_models}) ===")
    print(f"  'Context helps' (majority wrong on last_u1 but right on conv): {len(ch)} examples")
    print(f"    label=0: {(ch['label']==0).sum()}  label=1: {(ch['label']==1).sum()}")
    print(f"  'Context hurts' (majority right on last_u1 but wrong on conv): {len(cu)} examples")
    print(f"    label=0: {(cu['label']==0).sum()}  label=1: {(cu['label']==1).sum()}")



=== CSC (total models=8) ===
  'Context helps' (majority wrong on last_u1 but right on conv): 18 examples
    label=0: 15  label=1: 3
  'Context hurts' (majority right on last_u1 but wrong on conv): 4 examples
    label=0: 2  label=1: 2

=== MultiPICo (total models=8) ===
  'Context helps' (majority wrong on last_u1 but right on conv): 5 examples
    label=0: 5  label=1: 0
  'Context hurts' (majority right on last_u1 but wrong on conv): 0 examples
    label=0: 0  label=1: 0


## 5. Model-unique errors (errors made by only one model)

Which model makes errors that no other model makes? These are model-specific weaknesses.

In [7]:
for dataset in DATASETS:
    for inp in INPUT_TYPES:
        sub = all_preds[(all_preds["dataset"] == dataset) & (all_preds["input_type"] == inp)]
        # per example: count how many models got it wrong
        wrong_counts = (
            sub[sub["correct"] == 0]
            .groupby("unique_id")["model"]
            .nunique()
            .reset_index(name="n_wrong_models")
        )
        # unique errors = only 1 model wrong
        unique_errors = wrong_counts[wrong_counts["n_wrong_models"] == 1]["unique_id"].tolist()
        # which model is responsible?
        resp = (
            sub[(sub["correct"] == 0) & (sub["unique_id"].isin(unique_errors))]
            .groupby("model")["unique_id"]
            .count()
            .sort_values(ascending=False)
        )
        print(f"\n[{dataset} | {inp}]  unique errors per model:")
        print(resp.to_string())



[CSC | last_u1_message]  unique errors per model:
model
XLM-RoBERTa-large_fine_tuning     14
EuroBERT-210m_fine_tuning          8
Qwen3.5-9B_chain_of_thought        8
Gemma4-E4B_zero_shot               6
Gemma4-E4B_chain_of_thought        2
Ministral3-8B_chain_of_thought     1
Qwen3.5-9B_zero_shot               1

[CSC | conversation]  unique errors per model:
model
Ministral3-8B_zero_shot           16
Gemma4-E4B_zero_shot              10
Ministral3-8B_chain_of_thought     5
Qwen3.5-9B_chain_of_thought        5
EuroBERT-210m_fine_tuning          4
Qwen3.5-9B_zero_shot               4
Gemma4-E4B_chain_of_thought        3

[MultiPICo | last_u1_message]  unique errors per model:
model
Gemma4-E4B_chain_of_thought      4
XLM-RoBERTa-large_fine_tuning    4
Gemma4-E4B_zero_shot             3
EuroBERT-210m_fine_tuning        2
Ministral3-8B_zero_shot          2
Qwen3.5-9B_chain_of_thought      2

[MultiPICo | conversation]  unique errors per model:
model
Gemma4-E4B_zero_shot              9
Mi

## 6. Pairwise model disagreement matrix

For each pair of models (same dataset + input_type), what fraction of examples do they predict differently?

In [8]:
for dataset in DATASETS:
    for inp in INPUT_TYPES:
        sub = all_preds[(all_preds["dataset"] == dataset) & (all_preds["input_type"] == inp)]
        wide = sub.pivot_table(index="unique_id", columns="model", values="model_answer")
        wide = wide.dropna()
        model_cols = wide.columns.tolist()
        rows = []
        for m1, m2 in combinations(model_cols, 2):
            disagree_rate = (wide[m1] != wide[m2]).mean()
            rows.append({"model_A": m1, "model_B": m2, "disagree_rate": round(disagree_rate, 3)})
        df_dis = pd.DataFrame(rows).sort_values("disagree_rate", ascending=False)
        print(f"\n=== {dataset} | {inp} ===")
        print(df_dis.to_string(index=False))



=== CSC | last_u1_message ===
                       model_A                        model_B  disagree_rate
     EuroBERT-210m_fine_tuning  XLM-RoBERTa-large_fine_tuning          0.545
     EuroBERT-210m_fine_tuning    Qwen3.5-9B_chain_of_thought          0.529
          Qwen3.5-9B_zero_shot  XLM-RoBERTa-large_fine_tuning          0.510
       Ministral3-8B_zero_shot  XLM-RoBERTa-large_fine_tuning          0.500
          Gemma4-E4B_zero_shot  XLM-RoBERTa-large_fine_tuning          0.476
Ministral3-8B_chain_of_thought  XLM-RoBERTa-large_fine_tuning          0.474
     EuroBERT-210m_fine_tuning           Gemma4-E4B_zero_shot          0.457
   Gemma4-E4B_chain_of_thought  XLM-RoBERTa-large_fine_tuning          0.438
       Ministral3-8B_zero_shot    Qwen3.5-9B_chain_of_thought          0.435
   Qwen3.5-9B_chain_of_thought           Qwen3.5-9B_zero_shot          0.421
   Qwen3.5-9B_chain_of_thought  XLM-RoBERTa-large_fine_tuning          0.419
          Gemma4-E4B_zero_shot        Ministr

## 7. Qualitative inspection — load original dialogues and display hard cases

Load the original unified corpus to attach the actual dialogue text to the hard examples.

In [9]:
import json

CORPUS_ROOT = Path(r"c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\datasets")

def load_json_dataset(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    entries = data["entries"] if "entries" in data else data
    rows = []
    for e in entries:
        row = {
            "unique_id": e.get("unique_id"),
            "label":     e.get("label"),
            "post":      e.get("post"),
            "reply":     e.get("reply"),
            "context":   e.get("context"),
            "dialogue":  str(e.get("dialogue", "")),
            "source":    e.get("source"),
        }
        rows.append(row)
    return pd.DataFrame(rows)

csc_df       = load_json_dataset(CORPUS_ROOT / "CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_final.json")
multipico_df = load_json_dataset(CORPUS_ROOT / "MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_final.json")

corpus = {"CSC": csc_df, "MultiPICo": multipico_df}
print("CSC columns:",       csc_df.columns.tolist(),       "  rows:", len(csc_df))
print("MultiPICo columns:", multipico_df.columns.tolist(), "  rows:", len(multipico_df))


CSC columns: ['unique_id', 'label', 'post', 'reply', 'context', 'dialogue', 'source']   rows: 2905
MultiPICo columns: ['unique_id', 'label', 'post', 'reply', 'context', 'dialogue', 'source']   rows: 1377


In [10]:
def show_examples(uid_list, dataset_name, n=5, title=""):
    """Print dialogue text for a list of unique_ids."""
    df_src = corpus[dataset_name]
    # find text columns
    text_cols = [c for c in df_src.columns if c.lower() in ("post", "reply", "dialogue", "context", "text")]
    sample = uid_list[:n]
    print(f"\n{'='*70}")
    print(f"  {title}  [{dataset_name}]  showing {len(sample)}/{len(uid_list)}")
    print(f"{'='*70}")
    for uid in sample:
        row = df_src[df_src["unique_id"] == uid]
        if row.empty:
            print(f"\n  [{uid}]  (not found in corpus)")
            continue
        row = row.iloc[0]
        print(f"\n  ID: {uid}   label={row.get('label', '?')}")
        for c in text_cols:
            val = row.get(c, "")
            if pd.notna(val) and str(val).strip():
                print(f"  {c}: {str(val)[:300]}")
        print()


### 7a. All-models-wrong examples (conversation input)

In [11]:
for dataset in DATASETS:
    key = f"{dataset}_conversation"
    if key not in all_wrong_by_setting:
        continue
    uids = all_wrong_by_setting[key]["unique_id"].tolist()
    show_examples(uids, dataset, n=5, title=f"ALL MODELS WRONG | conversation")



  ALL MODELS WRONG | conversation  [CSC]  showing 5/14

  ID: 101_1035   label=None
  reply: Steve, should look of realistic resolutions, and stick to it, this is not great to keep breaking it
  context: Steve's new year's resolution was to learn French and go to the gym every day. That was three weeks ago.  At the moment, instead of that, he is watching sports and eating popcorn on the couch.
  dialogue: {'turns': [{'u1': "I know you wanted to start learning French and hit the gym every day, but, man, watching sports and eating popcorn sounds way better than slogging through grammar rules and getting a sweat going. Can't it be just one day a week, though?", 'u2': "That's exactly my point, though. If


  ID: 124_1011   label=None
  reply: great, which ones are clean?
  context: Steve says that your shirt is too tight for the occasion and says, "I can let you borrow one of mine if you want."
  dialogue: {'turns': [{'u1': "I think the shirt you're wearing is a bit too tight for the occa

### 7b. "Context helps" examples — models wrong on last_u1 but right with full conversation

In [12]:
for dataset in DATASETS:
    uids = context_helps[context_helps["dataset"] == dataset]["unique_id"].tolist()
    show_examples(uids, dataset, n=5, title="CONTEXT HELPS (lu1 wrong → conv right)")



  CONTEXT HELPS (lu1 wrong → conv right)  [CSC]  showing 5/18

  ID: 112_1046   label=None
  reply: Steve, it's for a fundraiser. Just go with it!
  context: Steve has to play soccer for a fundraising event he is a part of. He has always been bad at sports. He is complaining about having to play soccer soon.
  dialogue: {'turns': [{'u1': "Ugh, why me?! I suck at soccer. I'm gonna be the worst player ever. I'm basically a liability on the field.", 'u2': "Dude, everyone's gonna be bad at first. Just pick up the ball and kick it. It's not rocket science."}, {'u1': 'But what if I trip over my own feet? What if I get ki


  ID: 126_1017   label=None
  reply: That will be great!
  context: You are going to an office party that happens every year at a wine bar. Steve asks you how long the party is going to last.  Steve says, "I might join later if you are still there then."
  dialogue: {'turns': [{'u1': "I'm getting ready to head over to the wine bar. I was thinking of grabbing a drink and t

### 7c. "Context hurts" examples — models right on last_u1 but wrong with full conversation

In [13]:
for dataset in DATASETS:
    uids = context_hurts[context_hurts["dataset"] == dataset]["unique_id"].tolist()
    show_examples(uids, dataset, n=5, title="CONTEXT HURTS (lu1 right → conv wrong)")



  CONTEXT HURTS (lu1 right → conv wrong)  [CSC]  showing 4/4

  ID: 105_1042   label=None
  reply: Looking colourful, are we?
  context: Steve and you are hanging out tonight. He shows up wearing a red tank top, green shorts, and yellow sneakers.
  dialogue: {'turns': [{'u1': 'I like it, mate. That new tank top looks awesome.', 'u2': 'Just got it yesterday, thanks.'}, {'u1': "And the shorts aren't too green, right?", 'u2': "Haha, I think they're a bit too green, but I'm not complaining."}, {'u1': 'Looking good, friend!', 'u2': 'Looking colourful, are we


  ID: 222_2051   label=None
  reply: Enthralling stuff.
  context: Steve is a fan of Bulgarian folk music. Every week, he finds a different song and plays it on his phone and says, "I finally found one you'll like! This one is really good. Come on!"
  dialogue: {'turns': [{'u1': "I finally found one you'll like! This one is really good. Come on!", 'u2': 'Thanks for sharing, Steve!'}, {'u1': "You're going to love this one, trust me!",

## 8. Export hard-cases to CSV for manual inspection in the paper

In [14]:
OUT_DIR = Path(r"c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output")
OUT_DIR.mkdir(exist_ok=True)

def enrich_and_save(uid_df, dataset_name, filename):
    """Merge with corpus text columns and save."""
    df_src = corpus[dataset_name]
    text_cols = [c for c in df_src.columns if c.lower() in ("post", "reply", "dialogue", "context", "label", "unique_id")]
    merged = uid_df.merge(df_src[text_cols].drop_duplicates("unique_id"), on="unique_id", how="left")
    path = OUT_DIR / filename
    merged.to_csv(path, index=False)
    print(f"Saved {len(merged)} rows → {path}")
    return merged

def remove_stale_export(filename):
    """Remove an export left by an earlier run when its category is now empty."""
    path = OUT_DIR / filename
    if path.exists():
        path.unlink()
        print(f"Removed stale export: {path}")

for dataset in DATASETS:
    # all-models-wrong per input type
    for inp in INPUT_TYPES:
        key = f"{dataset}_{inp}"
        if key in all_wrong_by_setting and not all_wrong_by_setting[key].empty:
            enrich_and_save(
                all_wrong_by_setting[key][["unique_id", "label", "n_models_wrong"]],
                dataset,
                f"all_wrong_{dataset}_{inp}.csv"
            )
        else:
            remove_stale_export(f"all_wrong_{dataset}_{inp}.csv")

# context helps / hurts
for dataset in DATASETS:
    ch = context_helps[context_helps["dataset"] == dataset][["unique_id", "label", "lu1_wrong_conv_ok"]]
    if not ch.empty:
        enrich_and_save(ch, dataset, f"context_helps_{dataset}.csv")
    else:
        remove_stale_export(f"context_helps_{dataset}.csv")

    cu = context_hurts[context_hurts["dataset"] == dataset][["unique_id", "label", "lu1_ok_conv_wrong"]]
    if not cu.empty:
        enrich_and_save(cu, dataset, f"context_hurts_{dataset}.csv")
    else:
        remove_stale_export(f"context_hurts_{dataset}.csv")

print("\nAll exports done.")


Saved 27 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\all_wrong_CSC_last_u1_message.csv
Saved 14 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\all_wrong_CSC_conversation.csv
Saved 9 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\all_wrong_MultiPICo_last_u1_message.csv
Saved 6 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\all_wrong_MultiPICo_conversation.csv
Saved 18 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\context_helps_CSC.csv
Saved 4 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_